# Phase 3b — Multi-Head TCN Rainfall Prediction (TensorFlow)

**Project:** Deep Learning-Based Rainfall Data Imputation and Prediction System  
**Author:** Arinn Danish  
**Framework:** TensorFlow 2.21.0  
**Input:** `completed_daily_rainfall_cnn_tf.csv` (GAN Phase 2 output)  

## Architecture
- **5 TCN blocks** with dilations [1, 2, 4, 8, 16] — 96 filters each, causal padding, LayerNorm
- **3 separate output heads**: daily (next 1 day), weekly (7-day sum), monthly (30-day sum)
- **13 input features**: rainfall + 7/30-day rolling means + sin/cos seasonal encoding
- **3-seed ensemble** [42, 123, 456]

## Why Multi-Head?
Predicting a 30-day sequence and then aggregating it compounds errors — each predicted day
contributes its own error to the weekly/monthly aggregate. The multi-head approach predicts
each aggregate **directly** as a separate regression target, eliminating sequential error compounding.

## Results Summary
| Scale | NRMSE% | NMAE% | R²% | vs Old PyTorch |
|---|---|---|---|---|
| Daily (next 1d) | 239.0% | 97.3% | -3.22% | behind (+7.53%) |
| Weekly (7d sum) | 115.3% | 81.6% | +2.72% | **BEAT** (+1.31%) |
| Monthly (30d sum) | 67.1% | 51.2% | +23.75% | **BEAT** (+15.24%) |

## 1. Imports and Configuration

In [ ]:
import os, time
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_theme(style="whitegrid", font_scale=1.1)
print("TensorFlow:", tf.__version__)

# Configuration
STATIONS   = ["nada", "lembing", "reman"]
N_STATIONS = 3
LOOK_BACK  = 90    # days of history as input
BATCH_SIZE = 64
N_EPOCHS   = 300
PATIENCE   = 40    # early stopping
LR         = 1e-3  # initial learning rate (cosine decay to 1e-4)
SEEDS      = [42, 123, 456]  # ensemble seeds

## 2. Load GAN-Completed Data

In [ ]:
df = pd.read_csv("../data/processed/completed_daily_rainfall_cnn_tf.csv", parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)

print(f"Rows: {len(df)} | Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"NaN remaining: {df[STATIONS].isna().sum().sum()}")
df[STATIONS].describe().round(2)

## 3. Feature Engineering — Seasonal + Rolling Means

In [ ]:
# Seasonal encoding — captures Malaysia monsoon cycle without categorical features
doy = df["date"].dt.dayofyear.values
mth = df["date"].dt.month.values
df["sin_doy"]   = np.sin(2 * np.pi * doy / 365)
df["cos_doy"]   = np.cos(2 * np.pi * doy / 365)
df["sin_month"] = np.sin(2 * np.pi * mth / 12)
df["cos_month"] = np.cos(2 * np.pi * mth / 12)

# Rolling means — encode recent wetness state
for s in STATIONS:
    df[f"{s}_rm7"]  = df[s].rolling(7,  min_periods=1).mean()
    df[f"{s}_rm30"] = df[s].rolling(30, min_periods=1).mean()

FEAT_RAIN = STATIONS + [f"{s}_rm7" for s in STATIONS] + [f"{s}_rm30" for s in STATIONS]
FEAT_SEAS = ["sin_doy", "cos_doy", "sin_month", "cos_month"]
FEAT_ALL  = FEAT_RAIN + FEAT_SEAS
N_FEAT    = len(FEAT_ALL)
print(f"Total input features: {N_FEAT}")
print(FEAT_ALL)

## 4. Normalisation and Target Construction

In [ ]:
raw      = df[FEAT_ALL].values.astype("float32")
data_mm  = df[STATIONS].values.astype("float32")  # raw mm, for target construction

# Log-compress then MinMax scale the 9 rainfall/rolling features
rain_log  = np.log1p(raw[:, :len(FEAT_RAIN)])
rain_sc   = MinMaxScaler((0, 1))
rain_norm = rain_sc.fit_transform(rain_log).astype("float32")

# Seasonal features are already in [-1, 1]
seas_norm = raw[:, len(FEAT_RAIN):]
data_norm = np.concatenate([rain_norm, seas_norm], axis=1).astype("float32")

def inv_rain3(arr3):
    """Inverse transform: norm -> log -> mm for the 3 raw station columns."""
    n = arr3.reshape(-1, N_STATIONS).shape[0]
    buf = np.zeros((n, len(FEAT_RAIN)), "float32")
    buf[:, :N_STATIONS] = arr3.reshape(-1, N_STATIONS)
    return np.maximum(np.expm1(rain_sc.inverse_transform(buf))[:, :N_STATIONS], 0.0).reshape(arr3.shape)

# Build sequences and three separate targets
N       = len(data_norm)
MAX_SKIP = 30  # need 30 days ahead for monthly target
X_list, Yd_list, Yw_list, Ym_list = [], [], [], []

for i in range(LOOK_BACK, N - MAX_SKIP):
    X_list.append(data_norm[i-LOOK_BACK:i])          # (90, 13)
    Yd_list.append(data_norm[i, :N_STATIONS])         # next-day norm
    Yw_list.append(data_mm[i:i+7,  :].sum(axis=0))   # next 7d sum mm
    Ym_list.append(data_mm[i:i+30, :].sum(axis=0))   # next 30d sum mm

X_all  = np.array(X_list,  "float32")
Yd_all = np.array(Yd_list, "float32")
Yw_all = np.array(Yw_list, "float32")
Ym_all = np.array(Ym_list, "float32")

# Scale weekly/monthly targets to [0,1]
yw_sc = MinMaxScaler((0, 1)); Yw_norm = yw_sc.fit_transform(Yw_all).astype("float32")
ym_sc = MinMaxScaler((0, 1)); Ym_norm = ym_sc.fit_transform(Ym_all).astype("float32")

# Chronological 70/15/15 split
N_seq = len(X_all)
n_tr  = int(N_seq * 0.70)
n_va  = int(N_seq * 0.15)

Xtr,  Xva,  Xte  = X_all[:n_tr],    X_all[n_tr:n_tr+n_va],   X_all[n_tr+n_va:]
Ytr_d,Yva_d,Yte_d = Yd_all[:n_tr],  Yd_all[n_tr:n_tr+n_va],  Yd_all[n_tr+n_va:]
Ytr_w,Yva_w,Yte_w = Yw_norm[:n_tr], Yw_norm[n_tr:n_tr+n_va], Yw_norm[n_tr+n_va:]
Ytr_m,Yva_m,Yte_m = Ym_norm[:n_tr], Ym_norm[n_tr:n_tr+n_va], Ym_norm[n_tr+n_va:]

print(f"Train: {len(Xtr)} | Val: {len(Xva)} | Test: {len(Xte)}")
print(f"X shape: {X_all.shape} | Yd: {Yd_all.shape} | Yw: {Yw_all.shape} | Ym: {Ym_all.shape}")

## 5. Model Architecture — Multi-Head TCN

In [ ]:
def tcn_block(x, filters, ks, dr, drop=0.2):
    """Two dilated causal Conv1D layers with residual skip connection."""
    res = x
    for _ in range(2):
        x = tf.keras.layers.Conv1D(
            filters, ks, padding="causal", dilation_rate=dr, activation="relu",
            kernel_regularizer=tf.keras.regularizers.l2(1e-4),
            kernel_initializer="he_normal")(x)
        x = tf.keras.layers.LayerNormalization()(x)
        x = tf.keras.layers.Dropout(drop)(x)
    # Project residual if channel count changed
    if res.shape[-1] != filters:
        res = tf.keras.layers.Conv1D(filters, 1, padding="same")(res)
    return tf.keras.layers.Add()([res, x])


def build_model(seed=42):
    tf.random.set_seed(seed)
    np.random.seed(seed)

    inp = tf.keras.Input((LOOK_BACK, N_FEAT), name="inp")
    x   = inp

    # 5 TCN blocks with exponential dilation: 1, 2, 4, 8, 16
    for b in range(5):
        x = tcn_block(x, 96, 3, 2**b, drop=0.2)

    # Dual context: last timestep captures recent state, global avg captures long-term
    last = tf.keras.layers.Lambda(lambda t: t[:, -1, :])(x)
    gavg = tf.keras.layers.GlobalAveragePooling1D()(x)
    ctx  = tf.keras.layers.Concatenate()([last, gavg])
    ctx  = tf.keras.layers.Dense(192, activation="relu")(ctx)
    ctx  = tf.keras.layers.Dropout(0.2)(ctx)

    # Three separate output heads — no sequential rollout, no error compounding
    d_out = tf.keras.layers.Dense(N_STATIONS, name="daily")(ctx)
    w_out = tf.keras.layers.Dense(N_STATIONS, activation="sigmoid", name="weekly")(ctx)
    m_out = tf.keras.layers.Dense(N_STATIONS, activation="sigmoid", name="monthly")(ctx)

    return tf.keras.Model(inp, [d_out, w_out, m_out], name=f"TCN_MultiHead_s{seed}")


m = build_model(42)
print(f"Parameters: {m.count_params():,}")
m.summary()

## 6. Ensemble Training

In [ ]:
va_preds = []
te_preds = []
stop_eps = []
t0 = time.time()

for seed in SEEDS:
    print(f"\n--- Seed {seed} ---")
    model = build_model(seed)

    steps = N_EPOCHS * len(Xtr) // BATCH_SIZE
    lr_s  = tf.keras.optimizers.schedules.CosineDecay(LR, steps, alpha=1e-4)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr_s),
        loss={"daily": "mse", "weekly": "mse", "monthly": "mse"},
        loss_weights={"daily": 1.0, "weekly": 0.5, "monthly": 0.3}
    )

    es = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=PATIENCE,
        restore_best_weights=True, verbose=1)

    hist = model.fit(
        Xtr, {"daily": Ytr_d, "weekly": Ytr_w, "monthly": Ytr_m},
        validation_data=(Xva, {"daily": Yva_d, "weekly": Yva_w, "monthly": Yva_m}),
        epochs=N_EPOCHS, batch_size=BATCH_SIZE,
        callbacks=[es], verbose=0)

    ep  = len(hist.history["loss"])
    bvl = min(hist.history["val_loss"])
    stop_eps.append(ep)
    print(f"Stopped ep {ep} | best val_loss={bvl:.6f}")

    pd_va, pw_va, pm_va = model.predict(Xva, verbose=0)
    pd_te, pw_te, pm_te = model.predict(Xte, verbose=0)
    va_preds.append((pd_va, pw_va, pm_va))
    te_preds.append((pd_te, pw_te, pm_te))

print(f"\nTotal training time: {(time.time()-t0)/60:.1f} min")

## 7. Ensemble Averaging and Metric Evaluation

In [ ]:
def evaluate_all(preds_list, Yd_set, Yw_set_mm, Ym_set_mm, label):
    pd_ens = np.mean([p[0] for p in preds_list], axis=0)
    pw_ens = np.mean([p[1] for p in preds_list], axis=0)
    pm_ens = np.mean([p[2] for p in preds_list], axis=0)

    yt_d = inv_rain3(Yd_set)
    yp_d = inv_rain3(pd_ens)
    yt_w = Yw_set_mm
    yp_w = yw_sc.inverse_transform(pw_ens)
    yt_m = Ym_set_mm
    yp_m = ym_sc.inverse_transform(pm_ens)

    results = {}
    print(f"\n{'='*60}")
    print(f"  {label}")
    for scale, yt, yp in [("DAILY", yt_d, yp_d), ("WEEKLY", yt_w, yp_w), ("MONTHLY", yt_m, yp_m)]:
        print(f"\n  {scale}:")
        print(f"  {'Station':<12} {'NRMSE%':>8} {'NMAE%':>8} {'R2%':>8}")
        print(f"  {'-'*40}")
        all_t, all_p = [], []
        for j, s in enumerate(STATIONS):
            t = yt[:, j]; p = yp[:, j]
            rmse = np.sqrt(mean_squared_error(t, p))
            mae  = mean_absolute_error(t, p)
            r2   = r2_score(t, p)
            mo   = t.mean()
            print(f"  {s:<12} {rmse/mo*100:>7.1f}% {mae/mo*100:>7.1f}% {r2*100:>+7.2f}%")
            all_t.append(t); all_p.append(p)
        at = np.concatenate(all_t); ap = np.concatenate(all_p)
        rmse = np.sqrt(mean_squared_error(at, ap))
        mae  = mean_absolute_error(at, ap)
        r2   = r2_score(at, ap)
        mo   = at.mean()
        print(f"  {'OVERALL':<12} {rmse/mo*100:>7.1f}% {mae/mo*100:>7.1f}% {r2*100:>+7.2f}%")
        results[scale] = (yt, yp)
    return results


va_results = evaluate_all(
    va_preds, Yva_d,
    yw_sc.inverse_transform(Yva_w), ym_sc.inverse_transform(Yva_m),
    "VALIDATION SET")

te_results = evaluate_all(
    te_preds, Yte_d,
    yw_sc.inverse_transform(Yte_w), ym_sc.inverse_transform(Yte_m),
    "TEST SET")

## 8. Scatter Plots — Actual vs Predicted

In [ ]:
slabels = ["Ldg. Nada", "Sg. Lembing", "Ldg. Kuala Reman"]

for scale_tag, (yt, yp), fname in [
    ("Daily — Next 1 Day",   te_results["DAILY"],   "../figures/tcn/tcn_mh_scatter_daily.png"),
    ("Weekly — 7-Day Sum",   te_results["WEEKLY"],  "../figures/tcn/tcn_mh_scatter_weekly.png"),
    ("Monthly — 30-Day Sum", te_results["MONTHLY"], "../figures/tcn/tcn_mh_scatter_monthly.png"),
]:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, j, label in zip(axes, range(N_STATIONS), slabels):
        t = yt[:, j]; p = yp[:, j]
        r2   = r2_score(t, p)
        rmse = np.sqrt(mean_squared_error(t, p))
        mo   = t.mean()
        ax.scatter(t, p, alpha=0.25, s=6, color="steelblue")
        mv = max(t.max(), p.max()) * 1.05
        ax.plot([0, mv], [0, mv], "r--", lw=1, label="1:1")
        ax.set_title(f"{label}\nNRMSE={rmse/mo*100:.1f}%  R²={r2*100:.1f}%", fontweight="bold")
        ax.set_xlabel("Actual (mm)"); ax.set_ylabel("Predicted (mm)")
        ax.legend()
    plt.suptitle(f"Multi-Head TCN — {scale_tag} (Test Set)",
                 fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {fname}")

## 9. R²% Comparison — All Models

In [ ]:
models_bar = [
    "Old PyTorch\n(Daily)", "Old PyTorch\n(Weekly)", "Old PyTorch\n(Monthly)",
    "TF Basic\n(Daily-14)",
    "TF MultiHead\n(Daily)", "TF MultiHead\n(Weekly)", "TF MultiHead\n(Monthly)"
]
r2_bar = [7.53, 1.31, 15.24, -10.04, -3.22, 2.72, 23.75]
colors = ["#90A4AE", "#90A4AE", "#90A4AE",
          "#EF9A9A",
          "#42A5F5", "#66BB6A", "#FFA726"]

fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.bar(models_bar, r2_bar, color=colors, alpha=0.85, edgecolor="white", linewidth=1.2)
ax.axhline(0, color="black", lw=1.2, ls="--")
for bar, val in zip(bars, r2_bar):
    offset = 1.5 if val >= 0 else -3.5
    ax.text(bar.get_x() + bar.get_width()/2, val + offset,
            f"{val:+.2f}%", ha="center", va="bottom", fontsize=9, fontweight="bold")
ax.set_ylabel("R²%")
ax.set_ylim(min(r2_bar) - 8, max(r2_bar) + 12)
ax.set_title("R²% Comparison: Old PyTorch vs TF Basic vs TF Multi-Head Ensemble",
             fontweight="bold")
plt.tight_layout()
plt.savefig("../figures/tcn/tcn_r2_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Save Test Predictions

In [ ]:
yt_d, yp_d = te_results["DAILY"]
yt_w, yp_w = te_results["WEEKLY"]
yt_m, yp_m = te_results["MONTHLY"]

test_dates = df["date"].values[LOOK_BACK + n_tr + n_va : LOOK_BACK + n_tr + n_va + len(Xte)]
rows = []
for i in range(len(Xte)):
    row = {"date": str(test_dates[i])}
    for j, s in enumerate(STATIONS):
        row[f"{s}_daily_actual"]     = float(yt_d[i, j])
        row[f"{s}_daily_predicted"]  = float(yp_d[i, j])
        row[f"{s}_weekly_actual"]    = float(yt_w[i, j])
        row[f"{s}_weekly_predicted"] = float(yp_w[i, j])
        row[f"{s}_monthly_actual"]   = float(yt_m[i, j])
        row[f"{s}_monthly_predicted"]= float(yp_m[i, j])
    rows.append(row)

pd.DataFrame(rows).to_csv("../predictions/tcn_multihead_test_predictions.csv", index=False)
print(f"Saved: tcn_multihead_test_predictions.csv ({len(rows)} rows)")

## 11. Results Summary

### Test Set — Multi-Head TCN (3-seed ensemble)

| Scale | NRMSE% | NMAE% | R²% | vs Old PyTorch |
|---|---|---|---|---|
| Daily (next 1d) | 239.0% | 97.3% | -3.22% | behind (+7.53%) |
| Weekly (7d sum) | 115.3% | 81.6% | **+2.72%** | **BEAT** (+1.31%) |
| Monthly (30d sum) | 67.1% | 51.2% | **+23.75%** | **BEAT** (+15.24%) |

### Training
| Seed | Stop epoch | Best val_loss |
|---|---|---|
| 42  | 149 | 0.078143 |
| 123 | 141 | 0.079443 |
| 456 | 150 | 0.079775 |

Total training time: **~278 min** (CPU-only)

### Key Insight
The multi-head architecture directly predicts weekly and monthly aggregates without sequential error compounding, which is why it outperforms the 30-day rollout approach on those scales.
Monthly rainfall prediction (R²=+23.75%) is useful for water resource management, reservoir planning, and flood-season preparation in Pahang.

---

## 12. Enhancement: Hurdle Model + Extreme-Event Weighted Loss

### Motivation
Two structural problems in the base multi-head model:
1. **~47.5% dry days** — MSE pushes predictions toward zero, making wet-day errors invisible
2. **Heavy-rainfall underweighting** — extreme events dominate real impact but are sparse

### Approach
- **4th head** — binary wet/dry classifier (BCE loss) on shared backbone
- **Hurdle gating** — daily prediction zeroed when `wet_prob < 0.4`
- **Weighted MSE** — `w = 1 + 3·y²` on daily head (quadratic amplification on heavy rain)
- Look-back reduced 90 → 45 days (SHAP showed signal near zero after 30 days)

In [ ]:
from sklearn.metrics import accuracy_score

LOOK_BACK_H = 45   # reduced from 90 based on SHAP lag-correlation analysis
ALPHA_W     = 3.0  # quadratic weight amplifier for heavy-rain events
WET_THRESH  = 0.5  # mm threshold for wet/dry label

# ── Rebuild sequences with 45-day look-back ──────────────────────────────────
data_norm_base = data_norm   # the 13-feature array from Section 4
N_base = len(data_norm_base)
MAX_SKIP = 30

X_h, Yd_h, Yw_h, Ym_h, Yb_h = [], [], [], [], []
for i in range(LOOK_BACK_H, N_base - MAX_SKIP):
    X_h.append(data_norm_base[i-LOOK_BACK_H:i])
    Yd_h.append(data_norm_base[i, :N_STATIONS])
    Yw_h.append(data_mm[i:i+7,  :].sum(axis=0))
    Ym_h.append(data_mm[i:i+30, :].sum(axis=0))
    Yb_h.append((data_mm[i, :] > WET_THRESH).astype("float32"))

X_h  = np.array(X_h,  "float32")
Yd_h = np.array(Yd_h, "float32")
Yw_h = np.array(Yw_h, "float32")
Ym_h = np.array(Ym_h, "float32")
Yb_h = np.array(Yb_h, "float32")

yw_sc_h = MinMaxScaler((0,1)); Yw_hn = yw_sc_h.fit_transform(Yw_h).astype("float32")
ym_sc_h = MinMaxScaler((0,1)); Ym_hn = ym_sc_h.fit_transform(Ym_h).astype("float32")

n_tr_h = int(len(X_h)*0.70); n_va_h = int(len(X_h)*0.15)
Xtr_h,  Xva_h,  Xte_h   = X_h[:n_tr_h],    X_h[n_tr_h:n_tr_h+n_va_h],   X_h[n_tr_h+n_va_h:]
Ytr_dh, Yva_dh, Yte_dh   = Yd_h[:n_tr_h],   Yd_h[n_tr_h:n_tr_h+n_va_h],  Yd_h[n_tr_h+n_va_h:]
Ytr_wh, Yva_wh, Yte_wh   = Yw_hn[:n_tr_h],  Yw_hn[n_tr_h:n_tr_h+n_va_h], Yw_hn[n_tr_h+n_va_h:]
Ytr_mh, Yva_mh, Yte_mh   = Ym_hn[:n_tr_h],  Ym_hn[n_tr_h:n_tr_h+n_va_h], Ym_hn[n_tr_h+n_va_h:]
Ytr_bh, Yva_bh, Yte_bh   = Yb_h[:n_tr_h],   Yb_h[n_tr_h:n_tr_h+n_va_h],  Yb_h[n_tr_h+n_va_h:]

Yte_dh_mm = inv_rain3(Yte_dh)
Yte_wh_mm = yw_sc_h.inverse_transform(Yte_wh)
Yte_mh_mm = ym_sc_h.inverse_transform(Yte_mh)
print(f"Hurdle sequences: Train={len(Xtr_h)} Val={len(Xva_h)} Test={len(Xte_h)}")
print(f"Input shape: {X_h.shape}")

# ── Hurdle model: 4-head ─────────────────────────────────────────────────────
def tcn_block_h(x, filters, ks=3, dr=1, drop=0.2):
    res = x
    for _ in range(2):
        x = tf.keras.layers.Conv1D(
            filters, ks, padding="causal", dilation_rate=dr, activation="relu",
            kernel_regularizer=tf.keras.regularizers.l2(1e-4),
            kernel_initializer="he_normal")(x)
        x = tf.keras.layers.LayerNormalization()(x)
        x = tf.keras.layers.Dropout(drop)(x)
    if res.shape[-1] != filters:
        res = tf.keras.layers.Conv1D(filters, 1, padding="same")(res)
    return tf.keras.layers.Add()([res, x])

def build_hurdle_model(seed=42, look_back=LOOK_BACK_H, n_feat=13):
    tf.random.set_seed(seed); np.random.seed(seed)
    inp = tf.keras.Input((look_back, n_feat))
    x = inp
    for b in range(5):
        x = tcn_block_h(x, 96, 3, 2**b)
    last = tf.keras.layers.Lambda(lambda t: t[:,-1,:])(x)
    gavg = tf.keras.layers.GlobalAveragePooling1D()(x)
    ctx  = tf.keras.layers.Concatenate()([last, gavg])
    ctx  = tf.keras.layers.Dense(256, activation="relu")(ctx)
    ctx  = tf.keras.layers.Dropout(0.2)(ctx)
    ctx  = tf.keras.layers.Dense(128, activation="relu")(ctx)
    ctx  = tf.keras.layers.Dropout(0.1)(ctx)
    b_out = tf.keras.layers.Dense(N_STATIONS, activation="sigmoid", name="wetdry")(ctx)
    d_out = tf.keras.layers.Dense(N_STATIONS, name="daily")(ctx)
    w_out = tf.keras.layers.Dense(N_STATIONS, activation="sigmoid", name="weekly")(ctx)
    m_out = tf.keras.layers.Dense(N_STATIONS, activation="sigmoid", name="monthly")(ctx)
    return tf.keras.Model(inp, [b_out, d_out, w_out, m_out])

def weighted_mse(y_true, y_pred):
    w = 1.0 + ALPHA_W * tf.square(y_true)
    return tf.reduce_mean(w * tf.square(y_true - y_pred))

mh = build_hurdle_model(42)
print(f"\nHurdle model parameters: {mh.count_params():,}")
del mh

# ── Train 3-seed ensemble ─────────────────────────────────────────────────────
va_preds_h, te_preds_h = [], []
t0 = time.time()
for seed in SEEDS:
    print(f"\nSeed {seed}", flush=True)
    model = build_hurdle_model(seed)
    steps = 300 * len(Xtr_h) // 64
    lr_s  = tf.keras.optimizers.schedules.CosineDecay(1e-3, steps, alpha=1e-4)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr_s),
        loss={"wetdry":"binary_crossentropy","daily":weighted_mse,"weekly":"mse","monthly":"mse"},
        loss_weights={"wetdry":1.0,"daily":1.0,"weekly":0.5,"monthly":0.3})
    es = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=50,
                                           restore_best_weights=True, verbose=1)
    model.fit(
        Xtr_h, {"wetdry":Ytr_bh,"daily":Ytr_dh,"weekly":Ytr_wh,"monthly":Ytr_mh},
        validation_data=(Xva_h,{"wetdry":Yva_bh,"daily":Yva_dh,"weekly":Yva_wh,"monthly":Yva_mh}),
        epochs=300, batch_size=64, callbacks=[es], verbose=0)
    print(f"  done — best val_loss={min(model.history.history['val_loss']):.5f}", flush=True)
    pb_va,pd_va,pw_va,pm_va = model.predict(Xva_h, verbose=0)
    pb_te,pd_te,pw_te,pm_te = model.predict(Xte_h, verbose=0)
    va_preds_h.append((pb_va,pd_va,pw_va,pm_va))
    te_preds_h.append((pb_te,pd_te,pw_te,pm_te))
    del model; tf.keras.backend.clear_session()

print(f"\nHurdle training time: {(time.time()-t0)/60:.1f} min")

# ── Evaluate ──────────────────────────────────────────────────────────────────
pb_te_h = np.mean([p[0] for p in te_preds_h], axis=0)
pd_te_h = np.mean([p[1] for p in te_preds_h], axis=0) * (pb_te_h > 0.4).astype("float32")
pw_te_h = np.mean([p[2] for p in te_preds_h], axis=0)
pm_te_h = np.mean([p[3] for p in te_preds_h], axis=0)

yp_dh = inv_rain3(pd_te_h)
yp_wh = yw_sc_h.inverse_transform(pw_te_h)
yp_mh = ym_sc_h.inverse_transform(pm_te_h)

for label, yt, yp in [("DAILY (hurdle)", Yte_dh_mm, yp_dh),
                       ("WEEKLY",         Yte_wh_mm, yp_wh),
                       ("MONTHLY",        Yte_mh_mm, yp_mh)]:
    at = np.concatenate([yt[:,j] for j in range(N_STATIONS)])
    ap = np.concatenate([yp[:,j] for j in range(N_STATIONS)])
    print(f"{label:20s}  R²={r2_score(at,ap)*100:+.2f}%  NRMSE={np.sqrt(mean_squared_error(at,ap))/at.mean()*100:.1f}%")

acc_h = accuracy_score(Yte_bh.flatten(), (pb_te_h>0.4).astype(int).flatten())
print(f"Wet/dry accuracy: {acc_h*100:.1f}%")

---

## 13. Enhancement: ERA5 Atmospheric Reanalysis Integration

### Why ERA5?
Rainfall without atmospheric context has a theoretical daily R² ceiling of ~10–25%.
ERA5 provides 9 atmospheric variables (temperature, humidity, pressure, wind, precipitable water)
that are the actual physical drivers of rainfall, expected to push the ceiling to 35–55%.

### ERA5 Variables Added (9 total)
| Variable | Description | Unit |
|---|---|---|
| `era5_t2m` | 2m air temperature | °C |
| `era5_d2m` | 2m dewpoint temperature | °C |
| `era5_sp` | Surface pressure | hPa |
| `era5_u10` | 10m u-wind component | m/s |
| `era5_v10` | 10m v-wind component | m/s |
| `era5_tcwv` | Total column water vapour | kg/m² |
| `era5_tp` | Total precipitation (ERA5 own) | mm |
| `era5_rh` | Relative humidity (derived) | % |
| `era5_ws` | Wind speed (derived) | m/s |

**Total input features: 22** (9 rain+rolling + 4 seasonal + 9 ERA5)

In [ ]:
import io, zipfile, netCDF4 as nc

# ── Load pre-processed ERA5 daily CSV (output of era5_pahang parsing) ─────────
era5_df = pd.read_csv("../data/processed/era5_pahang_daily.csv", parse_dates=["date"])
ERA5_VARS = [c for c in era5_df.columns if c.startswith("era5_")]
print(f"ERA5 loaded: {len(era5_df)} rows | variables: {ERA5_VARS}")

# ── Merge with rainfall dataset ───────────────────────────────────────────────
rain_df2 = pd.read_csv("../data/processed/completed_daily_rainfall_cnn_tf.csv", parse_dates=["date"])
rain_df2 = rain_df2.sort_values("date").reset_index(drop=True)
df2 = rain_df2.merge(era5_df[["date"] + ERA5_VARS], on="date", how="left")
df2[ERA5_VARS] = df2[ERA5_VARS].ffill().bfill()
print(f"Merged: {df2.shape} | NaN: {df2[ERA5_VARS].isna().sum().sum()}")

# ── Feature engineering (same as Section 3) ───────────────────────────────────
doy2 = df2["date"].dt.dayofyear.values; mth2 = df2["date"].dt.month.values
df2["sin_doy"]   = np.sin(2*np.pi*doy2/365); df2["cos_doy"]   = np.cos(2*np.pi*doy2/365)
df2["sin_month"] = np.sin(2*np.pi*mth2/12);  df2["cos_month"] = np.cos(2*np.pi*mth2/12)
for s in STATIONS:
    df2[f"{s}_rm7"]  = df2[s].rolling(7,  min_periods=1).mean()
    df2[f"{s}_rm30"] = df2[s].rolling(30, min_periods=1).mean()

FEAT_RAIN2 = STATIONS + [f"{s}_rm7" for s in STATIONS] + [f"{s}_rm30" for s in STATIONS]
FEAT_SEAS2 = ["sin_doy","cos_doy","sin_month","cos_month"]
FEAT_ALL2  = FEAT_RAIN2 + FEAT_SEAS2 + ERA5_VARS
N_FEAT2    = len(FEAT_ALL2)
print(f"Total features: {N_FEAT2}")

# ── Normalise ─────────────────────────────────────────────────────────────────
raw2      = df2[FEAT_ALL2].values.astype("float32")
n_rain2   = len(FEAT_RAIN2)
rain_log2 = np.log1p(raw2[:, :n_rain2])
rain_sc2  = MinMaxScaler((0,1)); rain_norm2 = rain_sc2.fit_transform(rain_log2).astype("float32")
seas_norm2 = raw2[:, n_rain2:n_rain2+len(FEAT_SEAS2)]
era5_sc2   = MinMaxScaler((0,1))
era5_norm2 = era5_sc2.fit_transform(raw2[:, n_rain2+len(FEAT_SEAS2):]).astype("float32")
data_norm2 = np.concatenate([rain_norm2, seas_norm2, era5_norm2], axis=1).astype("float32")
data_mm2   = df2[STATIONS].values.astype("float32")

def inv_daily2(arr):
    buf = np.zeros((len(arr), n_rain2), "float32")
    buf[:, :N_STATIONS] = np.clip(arr, 0, 1)
    return np.maximum(np.expm1(rain_sc2.inverse_transform(buf))[:, :N_STATIONS], 0.0)

# ── Build sequences (45-day look-back + wet/dry target) ───────────────────────
N2 = len(data_norm2)
X2_l, Yd2_l, Yw2_l, Ym2_l, Yb2_l = [], [], [], [], []
for i in range(45, N2 - 30):
    X2_l.append(data_norm2[i-45:i])
    Yd2_l.append(data_norm2[i, :N_STATIONS])
    Yw2_l.append(data_mm2[i:i+7, :].sum(axis=0))
    Ym2_l.append(data_mm2[i:i+30,:].sum(axis=0))
    Yb2_l.append((data_mm2[i, :] > 0.5).astype("float32"))

X2_all  = np.array(X2_l,  "float32"); Yd2_all = np.array(Yd2_l, "float32")
Yw2_all = np.array(Yw2_l, "float32"); Ym2_all = np.array(Ym2_l, "float32")
Yb2_all = np.array(Yb2_l, "float32")
yw_sc2  = MinMaxScaler((0,1)); Yw2_n = yw_sc2.fit_transform(Yw2_all).astype("float32")
ym_sc2  = MinMaxScaler((0,1)); Ym2_n = ym_sc2.fit_transform(Ym2_all).astype("float32")

n_tr2 = int(len(X2_all)*0.70); n_va2 = int(len(X2_all)*0.15)
Xtr2,  Xva2,  Xte2   = X2_all[:n_tr2],    X2_all[n_tr2:n_tr2+n_va2],   X2_all[n_tr2+n_va2:]
Ytr2_d,Yva2_d,Yte2_d = Yd2_all[:n_tr2],   Yd2_all[n_tr2:n_tr2+n_va2],  Yd2_all[n_tr2+n_va2:]
Ytr2_w,Yva2_w,Yte2_w = Yw2_n[:n_tr2],     Yw2_n[n_tr2:n_tr2+n_va2],    Yw2_n[n_tr2+n_va2:]
Ytr2_m,Yva2_m,Yte2_m = Ym2_n[:n_tr2],     Ym2_n[n_tr2:n_tr2+n_va2],    Ym2_n[n_tr2+n_va2:]
Ytr2_b,Yva2_b,Yte2_b = Yb2_all[:n_tr2],   Yb2_all[n_tr2:n_tr2+n_va2],  Yb2_all[n_tr2+n_va2:]

Yte2_d_mm = inv_daily2(Yte2_d)
Yte2_w_mm = yw_sc2.inverse_transform(Yte2_w)
Yte2_m_mm = ym_sc2.inverse_transform(Yte2_m)
print(f"Sequences: Train={len(Xtr2)} Val={len(Xva2)} Test={len(Xte2)} | Input={X2_all.shape[1:]}")

In [ ]:
# ── Train ERA5 + Hurdle ensemble (22 features, 45-day look-back) ──────────────
va_preds2, te_preds2 = [], []
t0 = time.time()
for seed in SEEDS:
    print(f"\nSeed {seed}", flush=True)
    model = build_hurdle_model(seed, look_back=45, n_feat=N_FEAT2)
    steps = 300 * len(Xtr2) // 64
    lr_s  = tf.keras.optimizers.schedules.CosineDecay(1e-3, steps, alpha=1e-4)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr_s),
        loss={"wetdry":"binary_crossentropy","daily":weighted_mse,"weekly":"mse","monthly":"mse"},
        loss_weights={"wetdry":1.0,"daily":1.0,"weekly":0.5,"monthly":0.3})
    es = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=50,
                                           restore_best_weights=True, verbose=1)
    model.fit(
        Xtr2, {"wetdry":Ytr2_b,"daily":Ytr2_d,"weekly":Ytr2_w,"monthly":Ytr2_m},
        validation_data=(Xva2,{"wetdry":Yva2_b,"daily":Yva2_d,"weekly":Yva2_w,"monthly":Yva2_m}),
        epochs=300, batch_size=64, callbacks=[es], verbose=0)
    print(f"  done", flush=True)
    pb_va,pd_va,pw_va,pm_va = model.predict(Xva2, verbose=0)
    pb_te,pd_te,pw_te,pm_te = model.predict(Xte2, verbose=0)
    va_preds2.append((pb_va,pd_va,pw_va,pm_va))
    te_preds2.append((pb_te,pd_te,pw_te,pm_te))
    del model; tf.keras.backend.clear_session()

print(f"\nERA5+Hurdle training time: {(time.time()-t0)/60:.1f} min")

# ── Evaluate & visualise ──────────────────────────────────────────────────────
pb2 = np.mean([p[0] for p in te_preds2], axis=0)
pd2 = np.mean([p[1] for p in te_preds2], axis=0) * (pb2 > 0.4).astype("float32")
pw2 = np.mean([p[2] for p in te_preds2], axis=0)
pm2 = np.mean([p[3] for p in te_preds2], axis=0)

yp2_d = inv_daily2(pd2);             yp2_w = yw_sc2.inverse_transform(pw2)
yp2_m = ym_sc2.inverse_transform(pm2)
acc2  = accuracy_score(Yte2_b.flatten(), (pb2>0.4).astype(int).flatten())

print(f"\n{'='*60}")
print("ERA5 + HURDLE TCN — TEST RESULTS")
print(f"{'='*60}")
for label, yt, yp in [("DAILY (hurdle)", Yte2_d_mm, yp2_d),
                       ("WEEKLY",         Yte2_w_mm, yp2_w),
                       ("MONTHLY",        Yte2_m_mm, yp2_m)]:
    at = np.concatenate([yt[:,j] for j in range(N_STATIONS)])
    ap = np.concatenate([yp[:,j] for j in range(N_STATIONS)])
    print(f"  {label:20s}  R²={r2_score(at,ap)*100:+.2f}%  NRMSE={np.sqrt(mean_squared_error(at,ap))/at.mean()*100:.1f}%")
print(f"  Wet/dry accuracy: {acc2*100:.1f}%")

# ── Scatter plots ─────────────────────────────────────────────────────────────
slabels = ["Ldg. Nada","Sg. Lembing","Ldg. Kuala Reman"]
for tag, yt, yp, fname in [
    ("Daily",   Yte2_d_mm, yp2_d, "../figures/tcn/era5_scatter_daily.png"),
    ("Monthly", Yte2_m_mm, yp2_m, "../figures/tcn/era5_scatter_monthly.png"),
]:
    fig, axes = plt.subplots(1,3,figsize=(18,5))
    for ax,j,label in zip(axes, range(N_STATIONS), slabels):
        t=yt[:,j]; p=yp[:,j]
        r2=r2_score(t,p); rmse=np.sqrt(mean_squared_error(t,p)); mo=t.mean()
        ax.scatter(t,p,alpha=0.3,s=8,color="steelblue")
        mv=max(t.max(),p.max())*1.05; ax.plot([0,mv],[0,mv],"r--",lw=1)
        ax.set_title(f"{label}\nNRMSE={rmse/mo*100:.1f}%  R²={r2:.4f}",fontweight="bold")
        ax.set_xlabel("Actual (mm)"); ax.set_ylabel("Predicted (mm)")
    plt.suptitle(f"ERA5+Hurdle TCN — {tag} (Test Set)",fontsize=13,fontweight="bold",y=1.02)
    plt.tight_layout(); plt.savefig(fname,dpi=150,bbox_inches="tight"); plt.show()

pd.DataFrame({
    "model":   ["Old PyTorch","Old PyTorch","Old PyTorch",
                 "TF Multi-Head","TF Multi-Head","TF Multi-Head",
                 "Hurdle (no ERA5)","Hurdle (no ERA5)","Hurdle (no ERA5)",
                 "ERA5+Hurdle","ERA5+Hurdle","ERA5+Hurdle"],
    "scale":   ["Daily","Weekly","Monthly"]*4,
    "r2":      [0.0753,0.0131,0.1524, -0.0322,0.0272,0.2375,
                 0.0284,0.0451,0.2114,
                 r2_score(np.concatenate([Yte2_d_mm[:,j] for j in range(3)]),
                          np.concatenate([yp2_d[:,j]     for j in range(3)])),
                 r2_score(np.concatenate([Yte2_w_mm[:,j] for j in range(3)]),
                          np.concatenate([yp2_w[:,j]     for j in range(3)])),
                 r2_score(np.concatenate([Yte2_m_mm[:,j] for j in range(3)]),
                          np.concatenate([yp2_m[:,j]     for j in range(3)]))]
}).to_csv("../predictions/era5_hurdle_predictions.csv", index=False)
print("\nSaved: era5_hurdle_predictions.csv")